# Language Models
:label:`sec_language-model`

In :numref:`sec_text-sequence`, we saw how to map text sequences into tokens, where these tokens can be viewed as a sequence of discrete observations such as words or characters. Assume that the tokens in a text sequence of length $T$ are in turn $x_1, x_2, \ldots, x_T$.
The goal of *language models*
is to estimate the joint probability of the whole sequence:

$$P(x_1, x_2, \ldots, x_T),$$

where statistical tools
in :numref:`sec_sequence`
can be applied.

Language models are incredibly useful. For instance, an ideal language model should generate natural text on its own, simply by drawing one token at a time $x_t \sim P(x_t \mid x_{t-1}, \ldots, x_1)$.
Quite unlike the monkey using a typewriter, all text emerging from such a model would pass as natural language, e.g., English text. Furthermore, it would be sufficient for generating a meaningful dialog, simply by conditioning the text on previous dialog fragments.
Clearly we are still very far from designing such a system, since it would need to *understand* the text rather than just generate grammatically sensible content.

Nonetheless, language models are of great service even in their limited form.
For instance, the phrases "to recognize speech" and "to wreck a nice beach" sound very similar.
This can cause ambiguity in speech recognition,
which is easily resolved through a language model that rejects the second translation as outlandish.
Likewise, in a document summarization algorithm
it is worthwhile knowing that "dog bites man" is much more frequent than "man bites dog", or that "I want to eat grandma" is a rather disturbing statement, whereas "I want to eat, grandma" is much more benign.


# 语言模型
:label:`sec_language-model`


在 :numref:`sec_text-sequence` 中，我们看到了如何将文本序列映射为词元，这些词元可以视为一系列离散的观测，例如单词或字符。假设长度为$T$的文本序列中的词元依次为$x_1, x_2, \ldots, x_T$。

*语言模型*的目标

是估计整个序列的联合概率：

$$P(x_1, x_2, \ldots, x_T),$$

其中 :numref:`sec_sequence` 中的统计工具

可以被应用。


语言模型非常有用。例如，一个理想的语言模型应该能够自主生成自然文本，只需通过每次采样一个词元$x_t \sim P(x_t \mid x_{t-1}, \ldots, x_1)$。

与使用打字机的猴子完全不同，从这种模型生成的所有文本都将作为自然语言（例如英语文本）通过。此外，仅通过将文本基于先前的对话片段进行条件化，就足以生成有意义的对话。

显然，我们距离设计这样的系统还很遥远，因为这样的系统需要真正*理解*文本，而不仅仅是生成语法合理的内容。


尽管如此，即使是以受限形式存在的语言模型仍然非常有用。

例如，短语"to recognize speech"（识别语音）和"to wreck a nice beach"（破坏一个美丽的海滩）发音非常相似。

这会导致语音识别中的歧义，

但通过语言模型可以轻松解决，因为语言模型会拒绝将第二种翻译视为荒谬的。

同样，在文档摘要算法中，

知道"狗咬人"比"人咬狗"更常见，或者"我想吃奶奶"是一个相当令人不安的陈述，而"我想吃，奶奶"则要温和得多，这些知识都非常有价值。


In [1]:
import torch
from d2l import torch as d2l

## Learning Language Models

The obvious question is how we should model a document, or even a sequence of tokens. 
Suppose that we tokenize text data at the word level.
Let's start by applying basic probability rules:

$$P(x_1, x_2, \ldots, x_T) = \prod_{t=1}^T P(x_t  \mid  x_1, \ldots, x_{t-1}).$$

For example, 
the probability of a text sequence containing four words would be given as:

$$\begin{aligned}&P(\textrm{deep}, \textrm{learning}, \textrm{is}, \textrm{fun}) \\
=&P(\textrm{deep}) P(\textrm{learning}  \mid  \textrm{deep}) P(\textrm{is}  \mid  \textrm{deep}, \textrm{learning}) P(\textrm{fun}  \mid  \textrm{deep}, \textrm{learning}, \textrm{is}).\end{aligned}$$

### Markov Models and $n$-grams
:label:`subsec_markov-models-and-n-grams`

Among those sequence model analyses in :numref:`sec_sequence`,
let's apply Markov models to language modeling.
A distribution over sequences satisfies the Markov property of first order if $P(x_{t+1} \mid x_t, \ldots, x_1) = P(x_{t+1} \mid x_t)$. Higher orders correspond to longer dependencies. This leads to a number of approximations that we could apply to model a sequence:

$$
\begin{aligned}
P(x_1, x_2, x_3, x_4) &=  P(x_1) P(x_2) P(x_3) P(x_4),\\
P(x_1, x_2, x_3, x_4) &=  P(x_1) P(x_2  \mid  x_1) P(x_3  \mid  x_2) P(x_4  \mid  x_3),\\
P(x_1, x_2, x_3, x_4) &=  P(x_1) P(x_2  \mid  x_1) P(x_3  \mid  x_1, x_2) P(x_4  \mid  x_2, x_3).
\end{aligned}
$$

The probability formulae that involve one, two, and three variables are typically referred to as *unigram*, *bigram*, and *trigram* models, respectively. 
In order to compute the language model, we need to calculate the
probability of words and the conditional probability of a word given
the previous few words.
Note that
such probabilities are
language model parameters.



### Word Frequency

Here, we
assume that the training dataset is a large text corpus, such as all
Wikipedia entries, [Project Gutenberg](https://en.wikipedia.org/wiki/Project_Gutenberg),
and all text posted on the
web.
The probability of words can be calculated from the relative word
frequency of a given word in the training dataset.
For example, the estimate $\hat{P}(\textrm{deep})$ can be calculated as the
probability of any sentence starting with the word "deep". A
slightly less accurate approach would be to count all occurrences of
the word "deep" and divide it by the total number of words in
the corpus.
This works fairly well, particularly for frequent
words. Moving on, we could attempt to estimate

$$\hat{P}(\textrm{learning} \mid \textrm{deep}) = \frac{n(\textrm{deep, learning})}{n(\textrm{deep})},$$

where $n(x)$ and $n(x, x')$ are the number of occurrences of singletons
and consecutive word pairs, respectively.
Unfortunately, 
estimating the
probability of a word pair is somewhat more difficult, since the
occurrences of "deep learning" are a lot less frequent. 
In particular, for some unusual word combinations it may be tricky to
find enough occurrences to get accurate estimates.
As suggested by the empirical results in :numref:`subsec_natural-lang-stat`,
things take a turn for the worse for three-word combinations and beyond.
There will be many plausible three-word combinations that we likely will not see in our dataset.
Unless we provide some solution to assign such word combinations a nonzero count, we will not be able to use them in a language model. If the dataset is small or if the words are very rare, we might not find even a single one of them.

### Laplace Smoothing

A common strategy is to perform some form of *Laplace smoothing*.
The solution is to
add a small constant to all counts. 
Denote by $n$ the total number of words in
the training set
and $m$ the number of unique words.
This solution helps with singletons, e.g., via

$$\begin{aligned}
	\hat{P}(x) & = \frac{n(x) + \epsilon_1/m}{n + \epsilon_1}, \\
	\hat{P}(x' \mid x) & = \frac{n(x, x') + \epsilon_2 \hat{P}(x')}{n(x) + \epsilon_2}, \\
	\hat{P}(x'' \mid x,x') & = \frac{n(x, x',x'') + \epsilon_3 \hat{P}(x'')}{n(x, x') + \epsilon_3}.
\end{aligned}$$

Here $\epsilon_1,\epsilon_2$, and $\epsilon_3$ are hyperparameters.
Take $\epsilon_1$ as an example:
when $\epsilon_1 = 0$, no smoothing is applied;
when $\epsilon_1$ approaches positive infinity,
$\hat{P}(x)$ approaches the uniform probability $1/m$. 
The above is a rather primitive variant of what
other techniques can accomplish :cite:`Wood.Gasthaus.Archambeau.ea.2011`.


Unfortunately, models like this get unwieldy rather quickly
for the following reasons. 
First, 
as discussed in :numref:`subsec_natural-lang-stat`,
many $n$-grams occur very rarely, 
making Laplace smoothing rather unsuitable for language modeling.
Second, we need to store all counts.
Third, this entirely ignores the meaning of the words. For
instance, "cat" and "feline" should occur in related contexts.
It is quite difficult to adjust such models to additional contexts,
whereas, deep learning based language models are well suited to
take this into account.
Last, long word
sequences are almost certain to be novel, hence a model that simply
counts the frequency of previously seen word sequences is bound to perform poorly there.
Therefore, we focus on using neural networks for language modeling
in the rest of the chapter.


## Perplexity
:label:`subsec_perplexity`

Next, let's discuss about how to measure the quality of the language model, which we will then use to evaluate our models in the subsequent sections.
One way is to check how surprising the text is.
A good language model is able to predict, with high accuracy, the tokens that come next.
Consider the following continuations of the phrase "It is raining", as proposed by different language models:

1. "It is raining outside"
1. "It is raining banana tree"
1. "It is raining piouw;kcj pwepoiut"

In terms of quality, Example 1 is clearly the best. The words are sensible and logically coherent.
While it might not quite accurately reflect which word follows semantically ("in San Francisco" and "in winter" would have been perfectly reasonable extensions), the model is able to capture which kind of word follows.
Example 2 is considerably worse by producing a nonsensical extension. Nonetheless, at least the model has learned how to spell words and some degree of correlation between words. Last, Example 3 indicates a poorly trained model that does not fit data properly.

We might measure the quality of the model by computing  the likelihood of the sequence.
Unfortunately this is a number that is hard to understand and difficult to compare.
After all, shorter sequences are much more likely to occur than the longer ones,
hence evaluating the model on Tolstoy's magnum opus
*War and Peace* will inevitably produce a much smaller likelihood than, say, on Saint-Exupery's novella *The Little Prince*. What is missing is the equivalent of an average.

Information theory comes handy here.
We defined entropy, surprisal, and cross-entropy
when we introduced the softmax regression
(:numref:`subsec_info_theory_basics`).
If we want to compress text, we can ask about
predicting the next token given the current set of tokens.
A better language model should allow us to predict the next token more accurately.
Thus, it should allow us to spend fewer bits in compressing the sequence.
So we can measure it by the cross-entropy loss averaged
over all the $n$ tokens of a sequence:

$$\frac{1}{n} \sum_{t=1}^n -\log P(x_t \mid x_{t-1}, \ldots, x_1),$$
:eqlabel:`eq_avg_ce_for_lm`

where $P$ is given by a language model and $x_t$ is the actual token observed at time step $t$ from the sequence.
This makes the performance on documents of different lengths comparable. For historical reasons, scientists in natural language processing prefer to use a quantity called *perplexity*. In a nutshell, it is the exponential of :eqref:`eq_avg_ce_for_lm`:

$$\exp\left(-\frac{1}{n} \sum_{t=1}^n \log P(x_t \mid x_{t-1}, \ldots, x_1)\right).$$

Perplexity can be best understood as the reciprocal of the geometric mean of the number of real choices that we have when deciding which token to pick next. Let's look at a number of cases:

* In the best case scenario, the model always perfectly estimates the probability of the target token as 1. In this case the perplexity of the model is 1.
* In the worst case scenario, the model always predicts the probability of the target token as 0. In this situation, the perplexity is positive infinity.
* At the baseline, the model predicts a uniform distribution over all the available tokens of the vocabulary. In this case, the perplexity equals the number of unique tokens of the vocabulary. In fact, if we were to store the sequence without any compression, this would be the best we could do for encoding it. Hence, this provides a nontrivial upper bound that any useful model must beat.

## Partitioning Sequences
:label:`subsec_partitioning-seqs`

We will design language models using neural networks
and use perplexity to evaluate 
how good the model is at 
predicting the next token given the current set of tokens
in text sequences.
Before introducing the model,
let's assume that it
processes a minibatch of sequences with predefined length
at a time.
Now the question is how to [**read minibatches of input sequences and target sequences at random**].


Suppose that the dataset takes the form of a sequence of $T$ token indices in `corpus`.
We will
partition it
into subsequences, where each subsequence has $n$ tokens (time steps).
To iterate over 
(almost) all the tokens of the entire dataset 
for each epoch
and obtain all possible length-$n$ subsequences,
we can introduce randomness.
More concretely,
at the beginning of each epoch,
discard the first $d$ tokens,
where $d\in [0,n)$ is uniformly sampled at random.
The rest of the sequence
is then partitioned
into $m=\lfloor (T-d)/n \rfloor$ subsequences.
Denote by $\mathbf x_t = [x_t, \ldots, x_{t+n-1}]$ the length-$n$ subsequence starting from token $x_t$ at time step $t$. 
The resulting $m$ partitioned subsequences
are 
$\mathbf x_d, \mathbf x_{d+n}, \ldots, \mathbf x_{d+n(m-1)}.$
Each subsequence will be used as an input sequence into the language model.


For language modeling,
the goal is to predict the next token based on the tokens we have seen so far; hence the targets (labels) are the original sequence, shifted by one token.
The target sequence for any input sequence $\mathbf x_t$
is $\mathbf x_{t+1}$ with length $n$.

![Obtaining five pairs of input sequences and target sequences from partitioned length-5 subsequences.](../img/lang-model-data.svg) 
:label:`fig_lang_model_data`

:numref:`fig_lang_model_data` shows an example of obtaining five pairs of input sequences and target sequences with $n=5$ and $d=2$.


交叉熵（Cross Entropy）是信息论中衡量两个概率分布差异的重要指标，在机器学习中广泛用作分类任务的损失函数。

**数学定义**：
给定真实分布$P$和估计分布$Q$，交叉熵定义为：
$$H(P, Q) = -\mathbb{E}_{x \sim P} \log Q(x)$$

**关键特性**：
1. 与KL散度的关系：$H(P,Q)=H(P)+D_{\mathrm{KL}}(P \| Q)$
   - $H(P)$是真实分布的熵
   - $D_{\mathrm{KL}}$是KL散度，反映分布间差异
2. 在分类任务中：
   - $P$是标签的one-hot分布（真实分布）
   - $Q$是模型的softmax输出（预测分布）
3. 当预测完全准确时（$Q=P$），交叉熵等于熵

**在语言模型中的应用**：
1. 用于计算序列的平均负对数似然：
$$\frac{1}{n} \sum_{t=1}^n -\log P(x_t | x_{<t})$$
2. 直接反映模型预测能力：
   - 较低交叉熵 → 更好预测能力
   - 较高交叉熵 → 更多预测不确定性
3. 与困惑度的关系：困惑度=exp(交叉熵)

**示例说明**：
对于二分类任务（猫/狗识别）：
- 真实标签P=[1,0]（猫）
- 模型预测Q=[0.8,0.2]
- 交叉熵 = -(1*log0.8 + 0*log0.2) = 0.223

**实际意义**：
- 梯度友好性：log函数导数有利于梯度传播
- 概率解释：直接优化预测概率与真实分布的匹配程度
- 多分类扩展：自然扩展到多类别场景（如词表预测）

详细数学推导见 :numref:`subsec_info_theory_basics`，在softmax回归中建立了交叉熵与极大似然估计的理论联系。

## 语言模型的学习

显而易见的问题是如何对一个文档（甚至是一个词元序列）进行建模。假设在单词级别对文本数据进行词元化。我们可以依靠基本的概率规则：

$$P(x_1, x_2, \ldots, x_T) = \prod_{t=1}^T P(x_t  \mid  x_1, \ldots, x_{t-1}).$$

例如，包含四个单词的文本序列的概率可以分解为：

$$\begin{aligned}&P(\textrm{deep}, \textrm{learning}, \textrm{is}, \textrm{fun}) \\
=&P(\textrm{deep}) P(\textrm{learning}  \mid  \textrm{deep}) P(\textrm{is}  \mid  \textrm{deep}, \textrm{learning}) P(\textrm{fun}  \mid  \textrm{deep}, \textrm{learning}, \textrm{is}).\end{aligned}$$

### 马尔可夫模型与n元语法
:label:`subsec_markov-models-and-n-grams`

在 :numref:`sec_sequence` 中的序列模型分析基础上，我们将马尔可夫模型应用于语言建模。如果序列分布满足一阶马尔可夫性质，即$P(x_{t+1} \mid x_t, \ldots, x_1) = P(x_{t+1} \mid x_t)$，则可以得到以下近似：

$$
\begin{aligned}
P(x_1, x_2, x_3, x_4) &=  P(x_1) P(x_2) P(x_3) P(x_4),\\
P(x_1, x_2, x_3, x_4) &=  P(x_1) P(x_2  \mid  x_1) P(x_3  \mid  x_2) P(x_4  \mid  x_3),\\
P(x_1, x_2, x_3, x_4) &=  P(x_1) P(x_2  \mid  x_1) P(x_3  \mid  x_1, x_2) P(x_4  \mid  x_2, x_3).
\end{aligned}
$$

涉及一个、两个和三个变量的概率公式通常分别称为*一元语法*、*二元语法*和*三元语法*模型。为了计算语言模型，需要统计单词出现概率及给定前几个单词时的条件概率。

### 词频统计

假设训练数据集是大型文本语料库（如所有维基百科条目、[古登堡计划](https://en.wikipedia.org/wiki/Project_Gutenberg)和网络文本）。单词概率可通过相对词频计算。例如，$\hat{P}(\textrm{deep})$可以估计为以"deep"开头的句子概率。更简单的方法是统计"deep"出现次数并除以语料库总词数。

对于条件概率，例如：

$$\hat{P}(\textrm{learning} \mid \textrm{deep}) = \frac{n(\textrm{deep, learning})}{n(\textrm{deep})},$$

其中$n(x)$和$n(x, x')$分别是单个词和连续词对的出现次数。但罕见词对的估计较困难，特别是三元组及更长组合可能未见诸训练数据。

### 拉普拉斯平滑

常用策略是进行*拉普拉斯平滑*，即在所有计数中添加小常数。设$n$为训练集总词数，$m$为唯一词数：

$$\begin{aligned}
	\hat{P}(x) & = \frac{n(x) + \epsilon_1/m}{n + \epsilon_1}, \\
	\hat{P}(x' \mid x) & = \frac{n(x, x') + \epsilon_2 \hat{P}(x')}{n(x) + \epsilon_2}, \\
	\hat{P}(x'' \mid x,x') & = \frac{n(x, x',x'') + \epsilon_3 \hat{P}(x'')}{n(x, x') + \epsilon_3}.
\end{aligned}$$

其中$\epsilon$为超参数。当$\epsilon_1 \to \infty$时，$\hat{P}(x)$趋近均匀分布$1/m$。这类方法存在存储开销大、忽略语义关联等缺陷，因此后续将重点介绍基于神经网络的语言模型。

## 困惑度
:label:`subsec_perplexity`

通过平均交叉熵损失衡量语言模型质量：

$$\frac{1}{n} \sum_{t=1}^n -\log P(x_t \mid x_{t-1}, \ldots, x_1),$$
:eqlabel:`eq_avg_ce_for_lm`

其指数形式称为*困惑度*：

$$\exp\left(-\frac{1}{n} \sum_{t=1}^n \log P(x_t \mid x_{t-1}, \ldots, x_1)\right).$$

困惑度可理解为选择下一个词时真实选项数的几何平均倒数：
- 最佳情况：模型完美预测（困惑度=1）
- 最差情况：总是预测0概率（困惑度→∞）
- 基线情况：均匀分布（困惑度=词表大小）

## 序列分区
:label:`subsec_partitioning-seqs`

使用神经网络设计语言模型时，需要将序列分区为固定长度的子序列。假设语料库有$T$个词元，每个epoch开始时随机丢弃前$d$个词元（$d \in [0,n)$），剩余序列划分为$m=\lfloor (T-d)/n \rfloor$个子序列。输入序列$\mathbf x_t = [x_t, \ldots, x_{t+n-1}]$，目标序列为$\mathbf x_{t+1}$。

![从长度5的子序列获得5组输入-目标序列对](../img/lang-model-data.svg) 
:label:`fig_lang_model_data`

:numref:`fig_lang_model_data`展示了$n=5$和$d=2$时的分区示例。

In [2]:
@d2l.add_to_class(d2l.TimeMachine)  #@save
def __init__(self, batch_size, num_steps, num_train=10000, num_val=5000):
    super(d2l.TimeMachine, self).__init__()
    self.save_hyperparameters()
    corpus, self.vocab = self.build(self._download())
    array = torch.tensor([corpus[i:i+num_steps+1]
                        for i in range(len(corpus)-num_steps)])
    self.X, self.Y = array[:,:-1], array[:,1:]

To train language models,
we will randomly sample 
pairs of input sequences and target sequences
in minibatches.
The following data loader randomly generates a minibatch from the dataset each time.
The argument `batch_size` specifies the number of subsequence examples in each minibatch
and `num_steps` is the subsequence length in tokens.

为训练语言模型，
我们将从小批量数据中随机采样
输入序列和目标序列对。
下面的数据加载器每次从数据集中随机生成一个小批量。
参数`batch_size`指定每个小批量中的子序列样本数量，
而`num_steps`是以词元为单位的子序列长度。

In [3]:
@d2l.add_to_class(d2l.TimeMachine)  #@save
def get_dataloader(self, train):
    idx = slice(0, self.num_train) if train else slice(
        self.num_train, self.num_train + self.num_val)
    return self.get_tensorloader([self.X, self.Y], train, idx)

As we can see in the following, 
a minibatch of target sequences
can be obtained 
by shifting the input sequences
by one token.


In [4]:
data = d2l.TimeMachine(batch_size=2, num_steps=10)
for X, Y in data.train_dataloader():
    print('X:', X, '\nY:', Y)
    break

X: tensor([[14,  2, 15,  0, 16, 22, 19,  0,  2, 15],
        [10, 15,  8,  0,  9, 10, 20,  0, 17,  2]]) 
Y: tensor([[ 2, 15,  0, 16, 22, 19,  0,  2, 15,  4],
        [15,  8,  0,  9, 10, 20,  0, 17,  2, 21]])


## Summary and Discussion

Language models estimate the joint probability of a text sequence. For long sequences, $n$-grams provide a convenient model by truncating the dependence. However, there is a lot of structure but not enough frequency to deal efficiently with infrequent word combinations via Laplace smoothing. Thus, we will focus on neural language modeling in subsequent sections.
To train language models, we can randomly sample pairs of input sequences and target sequences in minibatches. After training, we will use perplexity to measure the language model quality.

Language models can be scaled up with increased data size, model size, and amount in training compute. Large language models can perform desired tasks by predicting output text given input text instructions. As we will discuss later (e.g., :numref:`sec_large-pretraining-transformers`),
at the present moment
large language models form the basis of state-of-the-art systems across diverse tasks.


## Exercises

1. Suppose there are 100,000 words in the training dataset. How much word frequency and multi-word adjacent frequency does a four-gram need to store?
1. How would you model a dialogue?
1. What other methods can you think of for reading long sequence data?
1. Consider our method for discarding a uniformly random number of the first few tokens at the beginning of each epoch.
    1. Does it really lead to a perfectly uniform distribution over the sequences on the document?
    1. What would you have to do to make things even more uniform? 
1. If we want a sequence example to be a complete sentence, what kind of problem does this introduce in minibatch sampling? How can we fix it?


## 小结与讨论

语言模型用于估计文本序列的联合概率。对于长序列，$n$元语法通过截断依赖关系提供了一种便捷的建模方式。然而，这种模型虽然能捕捉局部结构，却难以有效处理低频词组合（即使采用拉普拉斯平滑）。因此，后续章节将重点探讨神经语言建模方法。

训练语言模型时，我们通过小批量随机采样输入序列与目标序列对。训练完成后，使用困惑度作为模型质量的评估指标。

语言模型的性能可通过扩大数据规模、模型参数量和计算资源实现显著提升。大语言模型能够根据输入指令预测输出文本，从而完成多样化任务（详见 :numref:`sec_large-pretraining-transformers`）。当前，大语言模型已成为各类任务最先进系统的核心基础。

## 练习题

1. 假设训练数据集包含100,000个单词，四元语法模型需要存储多少词频和相邻词组合频率？
1. 如何对对话进行建模？
1. 你能想到哪些其他处理长序列数据的方法？
1. 考虑我们在每个epoch开始时随机丢弃前几个词元的策略：
    1. 这种方法真的能实现文档序列的完全均匀分布吗？
    1. 如何改进才能使分布更加均匀？
1. 若要求每个序列样本必须是完整句子，这会给小批量采样带来什么问题？如何解决？

[Discussions](https://discuss.d2l.ai/t/118)


**问题解析**：
该题目考察对n-gram模型空间复杂度的理解。四元语法模型需要存储两种频率信息：
1. **四元组频率**：连续四个词$(w_{t-3},w_{t-2},w_{t-1},w_t)$的出现次数
2. **三元组频率**：前三个词$(w_{t-3},w_{t-2},w_{t-1})$的出现次数

**理论计算**（假设所有组合都可能出现）：
- 词表大小$V=10^5$
- 四元组数量：$V^4 = (10^5)^4 = 10^{20}$ 
- 三元组数量：$V^3 = (10^5)^3 = 10^{15}$
- 总存储量：$10^{20} + 10^{15} \approx 1.00001 \times 10^{20}$个参数

**实际应用中的挑战**：
1. 数据稀疏性：实际语料中有效组合远少于理论值
2. 存储不可行：即使每个参数仅需4字节存储，总需约$4 \times 10^{20}$字节（约400亿TB）
3. 计算困难：如此大规模参数无法进行有效概率估计

**解决方法**：
- 平滑技术（如Kneser-Ney平滑）
- 神经网络语言模型（通过分布式表示降低维度）
- 子词切分（如BPE算法减少词表大小）

**示例对比**：
- 英语Wikipedia语料（约30亿词）中：
  - 实际出现的三元组约$10^{12}$量级
  - 四元组约$10^{16}$量级
- 仍远超出常规存储能力

此问题揭示了传统n-gram模型的根本局限性，解释了为何需要转向神经网络语言模型。